# MinbarAI cloud translation server (Colab / Kaggle)

Runs the **entire translation stack** on the free GPU: Quran verse matcher, TranslateGemma 12B (Ollama), Helsinki MT, QE reranker — behind one HTTP endpoint. The mosque PC then only needs mic + VAD + the overlay.

**Setup**
- Colab: `Runtime > Change runtime type > T4 GPU`
- Kaggle: `Settings > Accelerator > GPU T4 x2` **and** `Settings > Internet > On`

**Run all cells.** The tunnel cell prints:
```
TRANSLATE_SERVER_URL=https://xxxx.trycloudflare.com
```
Copy it into `.env` at the MinbarAI project root on the mosque PC. The app health-checks every 30 s and falls back to the local stack when the session dies.

In [ ]:
# 1. Ollama + model (installer needs zstd on Kaggle images)
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, time
env = os.environ.copy()
env["OLLAMA_KEEP_ALIVE"] = "-1"
ollama_server = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(5)
!ollama pull translategemma:12b

In [ ]:
# 2. Project code + Python deps
!git clone --depth 1 https://github.com/Yacine-DH/MinbarAI.git /content/MinbarAI 2>/dev/null || git clone --depth 1 https://github.com/Yacine-DH/MinbarAI.git /kaggle/working/MinbarAI
import pathlib
ROOT = next(p for p in (pathlib.Path("/content/MinbarAI"), pathlib.Path("/kaggle/working/MinbarAI")) if p.exists())
%cd {ROOT}
!pip install -q fastapi uvicorn rapidfuzz sentence-transformers transformers sentencepiece ollama python-dotenv

In [ ]:
# 3. Start the translation server (12B on the GPU) and wait until healthy
import json, os, subprocess, time, urllib.request

env = os.environ.copy()
env["OLLAMA_MODEL_ID"] = "translategemma:12b"
api = subprocess.Popen(
    ["python", "-m", "uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", "8000"],
    env=env, cwd=str(ROOT),
)

for _ in range(60):
    time.sleep(5)
    try:
        h = json.load(urllib.request.urlopen("http://localhost:8000/health", timeout=5))
        print(h)
        if h["quran"] and h["gemma"] and h["rerank"]:
            break
    except Exception as e:
        print("waiting...", e)

# warm up the 12B so the first khutbah chunk is fast
body = json.dumps({"text": "\u0627\u0644\u062d\u0645\u062f \u0644\u0644\u0647 \u0631\u0628 \u0627\u0644\u0639\u0627\u0644\u0645\u064a\u0646 \u0646\u062d\u0645\u062f\u0647 \u0648\u0646\u0633\u062a\u0639\u064a\u0646\u0647"}).encode()
req = urllib.request.Request("http://localhost:8000/translate", data=body, headers={"Content-Type": "application/json"})
t0 = time.time()
print(json.load(urllib.request.urlopen(req, timeout=300)), f"({time.time()-t0:.1f}s)")

In [ ]:
# 4. Tunnel — copy the printed TRANSLATE_SERVER_URL line into .env on the mosque PC
import re, subprocess

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n" + "=" * 60)
        print("TRANSLATE_SERVER_URL=" + m.group(0))
        print("=" * 60)
        break

In [ ]:
# 5. Keep-alive — leave running during the khutbah
import time

while True:
    time.sleep(600)
    print("alive", time.strftime("%H:%M:%S"))